In [ ]:
import pandas as pd


In [ ]:
df= pd.read_csv("transactions.csv")


In [ ]:
df.head()


In [ ]:
df1=pd.read_csv("accounts.csv")


In [ ]:
df1.head()


In [ ]:
merged_df = df.merge(
    df1,
    left_on="SENDER_ACCOUNT_ID",
    right_on="ACCOUNT_ID",
    how="left",
    suffixes=("_TX", "_ACCOUNT")
)


In [ ]:
merged_df.head()


In [ ]:
merged_df["TX_BEHAVIOR_ID"].unique()


In [ ]:
merged_df["TIMESTAMP"].head(20)


In [ ]:
merged_df["TIMESTAMP"].dtype


In [ ]:
merged_df["TIMESTAMP"].nunique()


In [ ]:
merged_df["TIMESTAMP"].min(), merged_df["TIMESTAMP"].max()


In [ ]:
sorted(merged_df["TIMESTAMP"].unique())[:30]


In [ ]:
merged_df.groupby(
    ["SENDER_ACCOUNT_ID", "TIMESTAMP"]
).size()


In [ ]:
merged_df.groupby("SENDER_ACCOUNT_ID")["CUSTOMER_ID"].nunique().max()


In [ ]:
(
    "C_" + merged_df["SENDER_ACCOUNT_ID"].astype(str)
    == merged_df["CUSTOMER_ID"]
).value_counts()


In [ ]:
df_correct1 = merged_df.drop(columns=["COUNTRY","TX_TYPE","ACCOUNT_TYPE","ALERT_ID","ACCOUNT_ID","CUSTOMER_ID"])


In [ ]:
df_correct1.head()


In [ ]:
df_correct1.isna().sum()


In [ ]:
df_correct1 = df_correct1.rename(columns={
    "TX_ID": "transaction_id",
    "SENDER_ACCOUNT_ID": "sender_account_id",
    "RECEIVER_ACCOUNT_ID": "receiver_account_id",
    "TX_AMOUNT": "transaction_amount",
    "TIMESTAMP": "time_period",
    "IS_FRAUD_TX": "is_fraud_transaction",
    "INIT_BALANCE": "initial_balance",
    "IS_FRAUD_ACCOUNT": "is_fraud_account",
    "TX_BEHAVIOR_ID": "transaction_behavior_id"
})


In [ ]:
df_correct1.head()


In [ ]:
df_correct1.isna().sum()


In [ ]:
account_features = (
    df_correct1
    .groupby("sender_account_id")
    .agg(
        transaction_count=("transaction_id", "count"),
        total_transaction_amount=("transaction_amount", "sum"),
        average_transaction_amount=("transaction_amount", "mean"),
        max_transaction_amount=("transaction_amount", "max"),
        unique_receiver_count=("receiver_account_id", "nunique"),
        initial_balance=("initial_balance", "first"),
        transaction_behavior_id=("transaction_behavior_id", "first"),
        is_fraud_account=("is_fraud_account", "first")
    )
    .reset_index()
)


In [ ]:
account_features.head()


In [ ]:
df_correct1.groupby("sender_account_id")[
    ["initial_balance", "transaction_behavior_id", "is_fraud_account"]
].nunique().max()


In [ ]:
transactions_per_period = (
    df_correct1
    .groupby(["sender_account_id", "time_period"])
    .size()
    .reset_index(name="transactions_in_period")
)


In [ ]:
max_tx_period = (
    transactions_per_period
    .groupby("sender_account_id")["transactions_in_period"]
    .max()
    .reset_index(name="max_transactions_per_period")
)


In [ ]:
account_features = account_features.merge(
    max_tx_period,
    on="sender_account_id",
    how="left"
)


In [ ]:
active_periods = (
    df_correct1
    .groupby("sender_account_id")["time_period"]
    .nunique()
    .reset_index(name="active_period_count")
)


In [ ]:
account_features = account_features.merge(
    active_periods,
    on="sender_account_id",
    how="left"
)


In [ ]:
account_features["average_transactions_per_active_period"] = (
    account_features["transaction_count"] /
    account_features["active_period_count"]
)


In [ ]:
account_features["max_transaction_to_balance_ratio"] = (
    account_features["max_transaction_amount"] /
    account_features["initial_balance"].replace(0, pd.NA)
)


In [ ]:
account_features.head()


In [ ]:
account_features.shape


In [ ]:
account_features.isna().sum()


In [ ]:
account_features.dtypes


In [ ]:
account_features.describe()


In [ ]:
df_correct1["transaction_behavior_id"].value_counts().sort_index()


In [ ]:
account_features= account_features.drop(columns=["is_fraud_account"])


In [ ]:
account_features.head()


In [ ]:
import pandas as pd

account_features["event_timestamp"] = pd.Timestamp.now(tz="UTC")


In [ ]:
account_features.head()


In [ ]:
account_features.to_parquet(
    "account_features.parquet",
    index=False
)


In [ ]:
test_parquet = pd.read_parquet(
    "account_features.parquet"
)

test_parquet.head()


In [ ]:
test_parquet.shape


In [ ]:
test_parquet.isna().sum()


In [ ]:
import sys
import os

python_folder = os.path.dirname(sys.executable)
print(python_folder)


In [ ]:
import shutil
import os

feature_repo_path = "/Users/nyxajoon/aml_feast_demo/feature_repo"
data_path = os.path.join(feature_repo_path, "data")

shutil.copy(
    "account_features.parquet",
    os.path.join(data_path, "account_features.parquet")
)


In [ ]:
import pandas as pd

check_df = pd.read_parquet(
    "/Users/nyxajoon/aml_feast_demo/feature_repo/data/account_features.parquet"
)

print(check_df.columns.tolist())


In [ ]:
feature_definitions_code = '''
from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Int64, Float64


account = Entity(
    name="account",
    join_keys=["sender_account_id"],
    description="Bank account identified by sender account ID"
)


account_features_source = FileSource(
    name="account_features_source",
    path="data/account_features.parquet",
    timestamp_field="event_timestamp"
)


account_aml_features = FeatureView(
    name="account_aml_features",
    entities=[account],
    ttl=timedelta(days=3650),

    schema=[
        Field(name="transaction_count", dtype=Int64),
        Field(name="total_transaction_amount", dtype=Float64),
        Field(name="average_transaction_amount", dtype=Float64),
        Field(name="max_transaction_amount", dtype=Float64),
        Field(name="unique_receiver_count", dtype=Int64),
        Field(name="initial_balance", dtype=Float64),
        Field(name="transaction_behavior_id", dtype=Int64),
        Field(name="max_transactions_per_period", dtype=Int64),
        Field(name="active_period_count", dtype=Int64),
        Field(name="average_transactions_per_active_period", dtype=Float64),
        Field(name="max_transaction_to_balance_ratio", dtype=Float64),
    ],

    source=account_features_source,
    online=True,
)
'''

with open(
    "/Users/nyxajoon/aml_feast_demo/feature_repo/feature_definitions.py",
    "w"
) as file:
    file.write(feature_definitions_code)


In [ ]:
feature_repo_path = "/Users/nyxajoon/aml_feast_demo/feature_repo"


In [ ]:
import os

os.chdir(feature_repo_path)
print(os.getcwd())


In [ ]:
!{python_folder}/feast apply


In [ ]:
!{python_folder}/feast entities list


In [ ]:
!{python_folder}/feast feature-views list


In [ ]:
!{python_folder}/feast materialize --disable-event-timestamp


In [ ]:
from feast import FeatureStore

store = FeatureStore(
    repo_path="/Users/nyxajoon/aml_feast_demo/feature_repo"
)


In [ ]:
result = store.get_online_features(
    features=[
        "account_aml_features:transaction_count",
        "account_aml_features:total_transaction_amount",
        "account_aml_features:average_transaction_amount",
        "account_aml_features:max_transaction_amount",
        "account_aml_features:unique_receiver_count",
        "account_aml_features:initial_balance",
        "account_aml_features:transaction_behavior_id",
        "account_aml_features:max_transactions_per_period",
        "account_aml_features:active_period_count",
        "account_aml_features:average_transactions_per_active_period",
        "account_aml_features:max_transaction_to_balance_ratio",
    ],
    entity_rows=[
        {"sender_account_id": 1}
    ]
).to_dict()


In [ ]:
result


In [ ]:
import subprocess
import os
import time

feast_executable = os.path.join(python_folder, "feast")

feast_server = subprocess.Popen(
    [
        feast_executable,
        "serve",
        "--host", "127.0.0.1",
        "--port", "6566"
    ],
    cwd="/Users/nyxajoon/aml_feast_demo/feature_repo",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(3)

print("Feast server process:", feast_server.poll())


In [ ]:
import requests

response = requests.get(
    "http://127.0.0.1:6566/docs"
)

print(response.status_code)


In [ ]:
payload = {
    "features": [
        "account_aml_features:transaction_count",
        "account_aml_features:total_transaction_amount",
        "account_aml_features:average_transaction_amount",
        "account_aml_features:max_transaction_amount",
        "account_aml_features:unique_receiver_count",
        "account_aml_features:initial_balance",
        "account_aml_features:transaction_behavior_id",
        "account_aml_features:max_transactions_per_period",
        "account_aml_features:active_period_count",
        "account_aml_features:average_transactions_per_active_period",
        "account_aml_features:max_transaction_to_balance_ratio"
    ],
    "entities": {
        "sender_account_id": [1]
    }
}


In [ ]:
response = requests.post(
    "http://127.0.0.1:6566/get-online-features",
    json=payload
)

print(response.status_code)


In [ ]:
response.json()


In [ ]:
mcp_server_code = '''
import requests
from mcp.server import MCPServer


mcp = MCPServer("AML Feast MCP Server")


FEAST_URL = "http://127.0.0.1:6566"


@mcp.tool()
def get_account_aml_features(sender_account_id: int) -> dict:
    payload = {
        "features": [
            "account_aml_features:transaction_count",
            "account_aml_features:total_transaction_amount",
            "account_aml_features:average_transaction_amount",
            "account_aml_features:max_transaction_amount",
            "account_aml_features:unique_receiver_count",
            "account_aml_features:initial_balance",
            "account_aml_features:transaction_behavior_id",
            "account_aml_features:max_transactions_per_period",
            "account_aml_features:active_period_count",
            "account_aml_features:average_transactions_per_active_period",
            "account_aml_features:max_transaction_to_balance_ratio"
        ],

        "entities": {
            "sender_account_id": [sender_account_id]
        }
    }

    response = requests.post(
        f"{FEAST_URL}/get-online-features",
        json=payload,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    feature_names = data["metadata"]["feature_names"]
    results = data["results"]

    output = {}

    for feature_name, result in zip(feature_names, results):
        if result["statuses"][0] == "PRESENT":
            output[feature_name] = result["values"][0]
        else:
            output[feature_name] = None

    return output


if __name__ == "__main__":
    mcp.run()
'''

with open(
    "/Users/nyxajoon/aml_feast_demo/mcp_server.py",
    "w"
) as file:
    file.write(mcp_server_code)

print("MCP server file created.")
